# NoLA walkthrough: read it, run it, interpret it

Start with the first two sections. Run one cell at a time with **Shift+Enter**. Open this notebook from `~/nola-walkthrough` and use the **Robotics Research Lab** kernel (Python with CUDA). The code uses the unmodified upstream checkout at `caec2722` and the local ViT-B/32 weights; no Kaggle or training jobs are needed here.

**Code map:** [`train.sh`](upstream/train.sh) → [`train.py`](upstream/train.py) → [`trainers/NoLA.py`](upstream/trainers/NoLA.py). The entry point loads the dataset and trainer; the trainer builds text vectors, selects anchors, trains the DINO teacher, and then tunes the CLIP student. This notebook follows that order, not the repository's folder order.

## 1. Trace one photo through the original CLIP model

Read [`clip/clip.py`](upstream/clip/clip.py): `_transform()` prepares the pixels and `load()` returns the model, embedding width, and preprocessing transform. This fork's default loader uses CUDA even when passed `device="cpu"`.

In [ ]:
from pathlib import Path
import sys

import torch
from PIL import Image
from IPython.display import display

root = Path.home() / "nola-walkthrough"
sys.dont_write_bytecode = True
sys.path[:0] = [str(root / "upstream"), str(root / ".deps")]
from clip import clip

model, width, preprocess = clip.load("ViT-B/32", device="cuda", download_root=str(root / "weights"))
model.eval()
print(torch.cuda.get_device_name(), "| embedding width:", width)

In [ ]:
photo = Image.open(root / "dog.jpg")
display(photo.resize((330, 260)))
image = preprocess(photo).unsqueeze(0).cuda()
classes = ["dog", "wolf", "cat"]
prompts = [f"a photo of a {name}" for name in classes]
tokens = clip.tokenize(prompts).cuda()
print("Image batch:", tuple(image.shape), "| text tokens:", tuple(tokens.shape))

In [`clip/model.py`](upstream/clip/model.py), find `CLIP.forward()`. It encodes the image and the prompts, normalizes the embeddings, and computes scaled cosine similarities. The resulting row has one logit per prompt. `softmax(dim=-1)` compares the choices in that row, not all possible labels.

In [ ]:
with torch.inference_mode():
    logits, _ = model(image, tokens)
    probabilities = logits[0].softmax(dim=-1)

for prompt, logit, probability in zip(prompts, logits[0].tolist(), probabilities.tolist()):
    print(f"{prompt:22} logit={logit:6.2f}  probability={probability:7.2%}")

**Interpretation:** “Dog” scores highest among these choices. This is not an accuracy measurement. Replace `wolf` with `golden retriever` in the input cell and rerun the two cells. Which logits should stay the same, and why can their probabilities still change?

## 2. Class description embeddings (CDE): from a class name to a CLIP weight

### 2.1 Where do the descriptions come from?

The LLM **writes text**; CLIP's text encoder **creates the vectors**. The [paper](https://bmvc2025.bmva.org/proceedings/281/) says its main experiments reuse CuPL's GPT-3.5 descriptions. The checkout also has an offline [example generator](upstream/descriptions/generic_descriptions.py): for every class name it asks six variations of a dataset-specific question (e.g. “How can you identify a forest in an aerial photo?”). It requests ten short GPT-3.5 Turbo Instruct completions per question and writes a JSON dictionary from class name to list of descriptions. That script's requested count is **not** the count stored in the released EuroSAT file, and we don't assume it recreated that file. At runtime, [`gen_emb()`](upstream/trainers/NoLA.py) uses cached vectors if present; otherwise it reads the released description JSON. Neither path calls an LLM during training.

Two separate sets of templates are involved: **questions for the LLM** (offline) and **short class phrases for CLIP** (included with the LLM answers at embedding time).

In [ ]:
import ast
import json

upstream = root / "upstream"
release_descriptions = json.loads((upstream / "descriptions/generic/EuroSAT.json").read_text())
# In training this order comes from the dataset class IDs. Here we explicitly use
# the released JSON's order for this text-only demo and keep it with the matrix.
eurosat_classes = list(release_descriptions)

# Read the literal from the original source without importing the full trainer.
source = ast.parse((upstream / "utils/data_utils.py").read_text())
template_table = next(node.value for node in source.body if isinstance(node, ast.Assign)
                      and any(isinstance(t, ast.Name) and t.id == "ds_specific_templates"
                              for t in node.targets))
eurosat_templates = ast.literal_eval(template_table)["EuroSAT"]

focus = "Forest"
print(f"{len(eurosat_classes)} EuroSAT classes, in demo order: {eurosat_classes}")
print(f"{focus}: {len(release_descriptions[focus])} released descriptions + {len(eurosat_templates)} CLIP phrases")
print("Description:", release_descriptions[focus][0])
print("CLIP phrase:", eurosat_templates[0].format(focus))

### 2.2 Implement `gen_emb()`'s calculation

For each class, concatenate its descriptions with the six formatted CLIP phrases. Encode the complete list, normalize each row, average the rows, and normalize the mean. Each **text** gets the same weight; NoLA does not first average the descriptions and the templates separately and then give the two groups equal weight. Stack the class vectors as columns, exactly the orientation returned by `gen_emb()`: a batch of normalized image features `[B, 512]` can multiply the matrix `[512, 10]` to give `[B, 10]` similarities. The trainer scales these to logits.

In [ ]:
@torch.inference_mode()
def encode_class(class_name, descriptions_by_class, templates):
    texts = descriptions_by_class[class_name] + [
        template.format(class_name.replace("_", " ")) for template in templates
    ]
    tokens = clip.tokenize(texts).to(next(model.parameters()).device)
    embeddings = model.encode_text(tokens)
    unit_embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
    mean = unit_embeddings.mean(dim=0)
    return mean / mean.norm()


cde_weights = torch.stack([
    encode_class(name, release_descriptions, eurosat_templates)
    for name in eurosat_classes
], dim=1)
forest_column = eurosat_classes.index(focus)
released_forest = cde_weights[:, forest_column]
assert cde_weights.shape == (width, len(eurosat_classes))
assert torch.isfinite(cde_weights).all()
torch.testing.assert_close(cde_weights.float().norm(dim=0),
                           torch.ones(len(eurosat_classes), device=cde_weights.device),
                           atol=1e-3, rtol=0)
print("CDE weight matrix [feature, class]:", tuple(cde_weights.shape))
print(f"Column {forest_column} belongs to {focus}; L2 norm: {released_forest.norm().item():.3f}")

### 2.3 Generate new descriptions for one class, then embed them

The example below adapts the upstream question styles into six short requests for **visible** cues. The six answers were written by the assistant for this walkthrough and are saved in the cell so it runs offline. They are not the paper's text and are not fabricated API output. `generate_descriptions()` accepts a `complete(question) -> str` function: use the saved answers here, or pass your own LLM client to regenerate them. We ask for one completion per question here, rather than the upstream example's ten. The original descriptions and template list stay unchanged; this is a demonstration of changing the inputs, not an accuracy comparison.

In [ ]:
def description_questions(class_name):
    return [
        f"Describe {class_name} in a satellite photo.",
        f"Describe how {class_name} would appear in an aerial image.",
        f"What does {class_name} look like from a satellite?",
        f"How can you identify {class_name} in an aerial photo?",
        f"Describe the visible texture of {class_name} in a satellite photo.",
        f"Describe the boundaries of {class_name} in an aerial photo.",
    ]


def generate_descriptions(class_name, complete):
    requests = (q + " Give one short sentence containing only visible cues."
                for q in description_questions(class_name))
    descriptions = [" ".join(complete(q).split()) for q in requests]
    if not all(descriptions):
        raise ValueError("The description generator returned an empty answer.")
    return {class_name: descriptions}


# One saved answer to each Forest question, in order. No API request is made.
saved_forest_answers = iter([
    "A broad patch of dark green canopy has irregular edges and occasional small clearings.",
    "Closely packed tree crowns create a continuous, uneven carpet of green.",
    "The canopy appears as a mottled block of lighter and darker greens rather than regular crop rows.",
    "Look for contiguous textured canopy and narrow tracks that interrupt it.",
    "The surface has a fine mottled texture from overlapping crowns and small shadows.",
    "The tree line forms a ragged border where dense canopy meets open ground.",
])
fresh_descriptions = generate_descriptions(focus, lambda question: next(saved_forest_answers))
fresh_forest = encode_class(focus, fresh_descriptions, eurosat_templates)
cosine = torch.nn.functional.cosine_similarity(released_forest.float(), fresh_forest.float(), dim=0)
assert fresh_forest.shape == (width,) and torch.isfinite(fresh_forest).all()
print("Example question:", description_questions(focus)[0])
print("Saved answer:", fresh_descriptions[focus][0])
print(f"Fresh input: {len(fresh_descriptions[focus])} descriptions + {len(eurosat_templates)} CLIP phrases")
print(f"Released vs. fresh Forest vector cosine: {cosine.item():.4f} (1 means same direction)")

**Interpretation:** The fresh vector is still a CLIP vector for Forest, but its direction can change with the wording and with how many descriptions you mix in. Cosine measures how much the vectors align; it says nothing about which one classifies images better. No EuroSAT images, API calls, or training are needed for this section.

## 3. Anchors → DINO teacher · source-reading guide, not run here

In [`trainers/NoLA.py`](upstream/trainers/NoLA.py), follow `before_train()` → `select_top_k()` → `train_taal()`. In [`utils/model_utils.py`](upstream/utils/model_utils.py), `top_k_indices_per_class2()` turns CLIP probability rows into selected image IDs and pseudo-labels. The original can use second- or third-choice predictions when a class gets no first-choice anchors. `Taal` encodes images with frozen DINO; `AdapterMLP` learns from the selected features and CLIP pseudo-labels.

**What to inspect in the next run:** predicted-class counts, selected anchor IDs, and teacher errors on a fixed split. Keep anchor pseudo-label precision separate from teacher accuracy.

## 4. DINO teacher → prompted CLIP student · source-reading guide, not run here

Follow `parse_batch_train()` → `forward_backward()` → `incorporate_prompt()`. The teacher supplies hard labels for one view of an image; the CLIP student sees a strong augmentation. Read `setup_optim()` to see what the released code updates: visual prompts, classifier head, and matching visual normalization parameters. Then read `after_epoch()` for validation and checkpoint selection.

**What to inspect in the next run:** which parameters update and how student held-out predictions differ from the teacher's. Training loss alone measures fit to pseudo-labels, not correctness on held-out data.

## 5. Reproduce one result · not run here

Start with a bounded EuroSAT pilot to check the loader, GPU memory, and time per step. Then use the matching split, class order and settings for the paper comparison. Keep three results separate: a small pilot, evaluation of an author checkpoint, and a fresh training run. The [paper](https://bmvc2025.bmva.org/proceedings/281/) reports **40.6% CLIP and 73.5% NoLA** top-1 on EuroSAT in Table 1; those are reference values, not results from this notebook.

DNC, the coverage proposal, and the frozen Kaggle study are intentionally out of scope.